# Descriptive statistics with federated analytics
In this notebook, you will describe patient data from three hospitals with **[vantage6](https://vantage6.ai/)**.
You will calculate **descriptive statistics**, for example, counts, means, and medians.

The hospitals are not allowed to share the data of their patients.
So we do not collect the data in one place. Instead, the analysis goes to the data:

1. Each hospital calculates statistics on its own data. Here, we call these the **partial** results.
2. One hospital combines the partial results into one overall result. Here, we call this the **aggregate** result.

Only statistics leave the hospitals, never the records of individual patients.
This is the basic idea behind federated analytics and federated learning.

At the end of this notebook, you have had a look at:
- explaining the difference between a partial result and an aggregate result;
- starting an analysis at several organisations with vantage6;
- comparing the partial results with the aggregate result and explain the differences.

You do not need to have a lot of programming experience to follow this notebook.
Read the text, run each cell from top to bottom (click on a cell and press `Shift` + `Enter`), and look at the output.
The more technical code is located in the `src/` directory. You can look at it if you are interested, but you do not need to.

# Before you start
You need to do these steps **once, in a terminal** (not in this notebook).
A terminal is the command window of your computer: '*Terminal*' on macOS and Linux, '*PowerShell*' on Windows.

1. **Install and start Docker.**
Docker runs the server and the hospitals as small virtual computers on your own computer.
Install [Docker Desktop](https://www.docker.com/products/docker-desktop/) and make sure it is running.
On Linux, your user must be allowed to use Docker. If you are not sure, ask your tutor.
2. **Install the Python packages.**
You need Python 3.10 or newer. Go to the folder of this course and run:
   ```shell
   python -m venv .venv
   .venv\Scripts\activate         # (Windows)
   source .venv/bin/activate      # (macOS and Linux)
   pip install -r requirements.txt
   ```
3. **Start the vantage6 developer network.**
In the same terminal, run:
   ```shell
   python dev_network.py start
   ```
   The first time, this can take several minutes, because Docker has to download the software.

The developer network is a small, complete federated vantage6 network on your own computer: one server, three organisations (each with one *node*, the software that runs the analyses at the organisation) and a website where you can follow your tasks: <http://localhost:7600> (username `dev_admin`, password `password`).

_Why do we start the network in a terminal and not in this notebook?
Starting the network takes a long time, and it may ask for extra permissions on your computer.
The network also needs to keep running when you restart this notebook. A terminal is better for this._

When you have finished, stop the network with `python dev_network.py stop`.

# Connect to the server
This section tells the notebook where the vantage6 server is.
The developer network runs on your own computer, so the address is `localhost`.
Run the cell to check that the server is running.

If you see an error, go back to **Before you start**.

In [ ]:
from src.utils import check_server

VANTAGE6_CONFIG = {
    "server_url": "http://localhost",
    "server_port": 7601,
    "server_api": "/api",
}

check_server(VANTAGE6_CONFIG)

# Log in
This section logs you in to the server with the user account of the developer network.

In a real vantage6 network, you would also need a multi-factor-authentication code and a private key to encrypt the data.
The developer network is only for testing, developing, and demonstrations, so it does not use these.

In [ ]:
from src.vantage_client import authenticate

client = authenticate(VANTAGE6_CONFIG,
                      username="dev_admin",
                      password="password",
                      use_mfa=False,
                      use_encryption=False,
                      log_level="warn")
print("You are logged in.")

# Choose the data and the variables
Each organisation in the developer network is a hospital that treats patients with head and neck cancer.
Each hospital has the data of 400 patients.
The data is made up (synthetic), so it does not belong to real people.

The data has one row per patient, with these columns (variables):

| Variable                     | Meaning                                                                                                                                                                                 |
|------------------------------|-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `id`                         | a code for the patient                                                                                                                                                                  |
| `biological_sex`             | `female` or `male`                                                                                                                                                                      |
| `age_at_diagnosis`           | age in years when the cancer was found                                                                                                                                                  |
| `performance_status_ecog`    | how well the patient can do daily activities, from `0` (no problems) to `4` (in bed all day)                                                                                            |
| `overall_hpv_p16_status`     | whether the tumour was caused by the HPV virus (`positive` or `negative`); patients with an HPV-positive tumour usually live longer                                                     |
| `index_tumour_location`      | where the tumour is located: `oropharynx` (back of the throat), `larynx` (voice box) or `hypopharynx` (lower part of the throat)                                                        |
| `clin_t`, `clin_n`, `clin_m` | the stage of the cancer: the size of the tumour (T: `0` to `4`), the spread to lymph nodes (N: `0` to `3`) and the spread to other parts of the body (M: `0` or `1`); `x` means unknown |
| `event_overall_survival`     | whether the patient passed away during the follow-up: `1` (deceased) or `0` (alive)                                                                                                     |
| `overall_survival_in_days`   | the number of days from the diagnosis until they passed away or until the end of the follow-up                                                                                          |

Each dataset on a node has a name, called a *label*. The label of this data set is `survival`.

Next, you choose which variables to describe. For each variable, you give the **datatype**:
- `numerical`: numbers, for example, age.
You calculate the count, mean, standard deviation, and quartiles.
- `categorical`: groups, for example, sex.
You calculate the number of patients in each group.

In [ ]:
# The label of the data set on the nodes
DATABASE_LABEL = "survival"

# The variables to describe
VARIABLES_TO_DESCRIBE = {
    "age_at_diagnosis": {"datatype": "numerical"},
    "overall_survival_in_days": {"datatype": "numerical"},
    "biological_sex": {"datatype": "categorical"},
    "performance_status_ecog": {"datatype": "categorical"},
    "overall_hpv_p16_status": {"datatype": "categorical"},
    "index_tumour_location": {"datatype": "categorical"},
    "clin_t": {"datatype": "categorical"},
    "event_overall_survival": {"datatype": "categorical"},
}

# Select your organisations to include
Here you choose the organisations (hospitals) that take part in the analysis.
Run the next cell to see which organisations are available.

In [ ]:
from src.utils import show_organisations

show_organisations(client)

Put the numbers (IDs) of the organisations that you want to include in the list below.
We start with including all three organisations.

In [ ]:
# Pick the organisations to run the task on
ORGANISATIONS_TO_RUN_TASK = [1, 2, 3]

# Who is orchestrating?
One organisation combines (aggregates) the partial results.
We call this organisation the **aggregator**.
The aggregator sends the question to all selected organisations, waits for their partial results, and combines them into one aggregate result.

The aggregator is set in the file `src/__init__.py` and not in this notebook, so that it cannot be changed by accident.
In real federated networks, this sometimes can even be an organisation that is separate from the hospitals, so that hospitals have to only trust the organisation doing the processing.
In the developer network, the aggregator is organisation 1.

In [ ]:
from src import AGGREGATOR_ORGANISATION

print(f"The aggregator is organisation {AGGREGATOR_ORGANISATION}.")

# Run the partial descriptive tasks
In vantage6, a question to one or more organisations is called a **task**.
A task uses an **algorithm**: a small program that the nodes download and run on their own data.
We use the [descriptive statistics algorithm of the STRONG AYA federated consortium](https://github.com/STRONGAYA/v6-descriptive-statistics).

An algorithm can contain several functions, called *methods*.
This algorithm has two that we use:
- `partial_general_statistics`: calculates the statistics on the data of one organisation (the partial result);
- `central`: asks all organisations for their partial results and combines them (the aggregate result).

In this section, we send the partial task to each organisation separately so that we can see the partial result of each hospital.
Typically, you would not do this: the central task does it for you.
We do it here so that you can compare the partial results with the aggregate result later.

To respect privacy, the algorithm does not share results that are based on 10 patients or fewer.

The first time, the nodes need to download the algorithm. This can take a few minutes. You can follow the tasks on the website: <http://localhost:7600>.

In [ ]:
from src.descriptive_statistics import send_task

# Create a partial descriptive statistics task per organisation
partial_tasks = {}
for organisation_id in ORGANISATIONS_TO_RUN_TASK:
    partial_tasks[organisation_id] = send_task(
        client,
        organisations=[organisation_id],
        method="partial_general_statistics",
        arguments={"variables_to_describe": VARIABLES_TO_DESCRIBE},
        database_label=DATABASE_LABEL,
    )

# Run the aggregating (central) task
This section sends the central task to the aggregator only.
The aggregator then creates new partial tasks for all selected organisations by itself and combines their results.
Notice that we tell the central task which organisations to include.

In [ ]:
# Create a task for the aggregator to retrieve the aggregated descriptive data
central_task = send_task(
    client,
    organisations=[AGGREGATOR_ORGANISATION],
    method="central",
    arguments={"variables_to_describe": VARIABLES_TO_DESCRIBE,
               "organisations_to_include": ORGANISATIONS_TO_RUN_TASK},
    database_label=DATABASE_LABEL,
)

# Retrieve the results
This section waits until all tasks have finished and then collects the results.
The results arrive as JSON: a text format that computers use to exchange data.

The results are stored in `raw_results`, a Python *dictionary*: a list of pairs of a *key* and a *value*.
Here, the key is the organisation ID (or `"aggregate"` for the aggregate result), and the value is the result.

In [ ]:
from src.descriptive_statistics import collect_results

raw_results = collect_results(client, partial_tasks, central_task)

Have a look at the raw result of organisation 1.
Is it easy to read?

In [ ]:
raw_results[1]

# Transform the results to a readable table
You probably find that the raw results are not the easiest to read.
Each result contains two tables:
- a table for the **numerical** variables, with one row per variable and statistic (for example: `age_at_diagnosis`, `mean`, `64.1`);
- a table for the **categorical** variables, with one row per variable and group (for example: `biological_sex`, `female`, `106`).

The function `read_results` turns the raw results into these two tables and adds a column `source` that shows where each row comes from.
The function `make_table` then puts the results of all sources next to each other so that you can compare them easily.

In [ ]:
from src.descriptive_statistics import read_results, make_table

numerical_results, categorical_results = read_results(raw_results)

# Inspect the results
Run the cells below to see the tables.

What do you see? Can you identify differences between the results of the partial tasks and the aggregate result?

Some words that you will see:
- `count`: the number of patients with a value;
- `na`: the number of missing values;
- `mean`: the average;
- `std`: the standard deviation (how much the values vary around the mean).
- `q1`, `median` and `q3`: the values below which 25 %, 50 %, and 75 % of the patients are.
- `NaN`: there is no value. For example, a hospital did not share a group because it has fewer than 10 patients in that group.

In [ ]:
make_table(numerical_results)

In [ ]:
make_table(categorical_results)

Have a think about the following:
1. Compare the `count` of each hospital with the `count` of the aggregate result. What do you notice?
2. Look at the `na` rows. Which hospitals have missing values? What could be the reason?
3. Is the aggregate `mean` the same as the average of the three means of the hospitals? Why, or why not?
4. Look at `q1`, `median` and `q3`. Why are the aggregate values not always whole numbers? What does this tell you about how the aggregator calculates them?
5. Look at `clin_t`. Why do some hospitals show `NaN` for stage `1`? Is the aggregate count for stage `1` correct?

The hospitals do not have the same number of patients in each group.
To compare them fairly, it is often better to look at **percentages**:

In [ ]:
make_table(categorical_results, show="percentage")

# Visualise differences
Tables are precise, but figures make it easier to see differences quickly.

- **Figure 1** shows the mean and the median of each numerical variable.
Each variable has its own scale because age (in years) and survival (in days) are measured in different units.
- **Figure 2** shows the percentage of patients in each group of the categorical variables.
Missing values are not in the figure, but you can find them in the tables above.

Each colour is one source. The last bar in each group (violet) is always the aggregate result.

In [ ]:
from src.descriptive_statistics import plot_numerical, plot_categorical

plot_numerical(numerical_results)
plot_categorical(categorical_results)

# Do the organisations use the same groups?
You can only combine results if all organisations record the data in the same way.
We call this **interoperability**.
This is especially important in federated networks, as you cannot really see the data of other organisations.

The figure below shows the T stage (`clin_t`). There is one row per organisation and one column per group.
The number is the percentage of patients in that group.
A grey cell means that the organisation has no patients in that group, or does not share it because it has fewer than 10 patients.

Look at the figure and think about these questions:
1. Which groups does each organisation use? Which organisation is different?
2. Look at the aggregate row. Is the percentage for T stage `2` correct? Why not?
3. In the next notebook, you will use the T stage to study survival.  What could you do with the data of this organisation?
For example, leave it out of the analysis, or first translate its groups into the groups of the other organisations. What are the advantages and disadvantages of each choice?

In [ ]:
from src.descriptive_statistics import plot_groups_per_source

plot_groups_per_source(categorical_results, "clin_t")

# Additional aspects to think about
1. Which hospital treats the youngest patients? And which hospital has the most HPV-positive patients?
2. Is the aggregate result a good description of each hospital? Which hospital is described the worst by the aggregate result, and why would that happen?
3. The aggregator only receives statistics, not the data of individual patients. Which statistics could still tell something about one person? Do we need to respect the minimum of 10 patients?
4. Change `ORGANISATIONS_TO_RUN_TASK`, so that the organisation that records the T stage differently is left out, and run the notebook again from **Select your organisations to include**. Does the T stage look correct now? What else changes in the aggregate result?

# Next step
Continue with the notebook **`run-cox-ph.ipynb`**.
There, you use the data of the hospitals to study survival. Keep the developer network running.

Keep this notebook open: you will need your results in the next notebook.

# When you have finished
Stop the developer network in the terminal with:
```shell
python dev_network.py stop
```
You can start it again later with `python dev_network.py start`.
Your tasks are kept. To delete the network completely, use `python dev_network.py remove`.